In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
print("[STEP 1] Parameters received")
print("=" * 80)
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_keys        : '{merge_keys}'")
print(f"  load_type         : '{load_type}'")

In [0]:
source_table = f"ws_healthcare_cntrl_india.bronze.{table_name}"
source_filter = f"landing_timestamp = '{landing_timestamp}'"

print("=" * 80)
print("[STEP 2] Reading bronze data")
print("=" * 80)
print(f"  Source table : {source_table}")
print(f"  Filter       : {source_filter}")
print(f"  Equivalent query:\n    SELECT * FROM {source_table} WHERE {source_filter}")

bronze_df = spark.read.table(source_table).filter(source_filter)

print(f"  Columns ({len(bronze_df.columns)}): {bronze_df.columns}")
print(f"  Rows matching filter: {bronze_df.count()}")

In [0]:
from pyspark.sql import functions as F

target_table = f"ws_healthcare_cntrl_india.silver.{table_name}"
normalized_load_type = (load_type or "FULL").strip().upper()
merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

print("=" * 80)
print("[STEP 3] Preparing load configuration")
print("=" * 80)
print(f"  Target table         : {target_table}")
print(f"  Load type (raw)      : '{load_type}'")
print(f"  Load type (resolved) : {normalized_load_type}" + ("  (defaulted to FULL because load_type was empty)" if not (load_type or "").strip() else ""))
print(f"  Merge keys           : {merge_key_list}")

if normalized_load_type == "MERGE":
    print("  Validating merge keys...")
    if not merge_key_list: # checking if the list is empty, when empty raise below exception
        print("  ERROR: No merge keys supplied for MERGE load.")
        raise ValueError("merge_keys must be provided when load_type is 'MERGE'.")

    missing_merge_keys = [key for key in merge_key_list if key not in bronze_df.columns]
    if missing_merge_keys:
        print(f"  ERROR: Merge keys missing in source: {missing_merge_keys}")
        raise ValueError(f"merge_keys not found in source data: {missing_merge_keys}")
    print("  Merge keys validated: all present in source data.")

# Audit columns are added for every load type (FULL / APPEND / MERGE)
print("  Adding audit columns: insert_timestamp, updated_timestamp")
updated_df = (
    bronze_df.withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("updated_timestamp", F.current_timestamp())
)
print(f"  Final columns ({len(updated_df.columns)}): {updated_df.columns}")

In [0]:
print("=" * 80)
print(f"[STEP 4] Writing data to {target_table} using load type: {normalized_load_type}")
print("=" * 80)

if normalized_load_type == "FULL":
    print(f"  FULL load: overwriting {target_table}")
    print(f"  Equivalent query:\n    CREATE OR REPLACE TABLE {target_table} AS SELECT * FROM <bronze_df with audit columns>")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("  FULL load completed.")
elif normalized_load_type == "APPEND":
    print(f"  APPEND load: appending to {target_table}")
    print(f"  Equivalent query:\n    INSERT INTO {target_table} SELECT * FROM <bronze_df with audit columns>")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("  APPEND load completed.")
elif normalized_load_type == "MERGE":
    print(f"  Checking whether target table exists: {target_table}")
    if not spark.catalog.tableExists(target_table):
        print("  Target table does not exist -> creating it with an initial full load.")
        print(f"  Equivalent query:\n    CREATE TABLE {target_table} AS SELECT * FROM <bronze_df with audit columns>")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("  Initial load completed.")
    else:
        print("  Target table exists -> performing MERGE.")
        source_view = "silver_merge_source"
        updated_df.createOrReplaceTempView(source_view)
        print(f"  Registered source temp view: {source_view}")

        merge_condition = " AND ".join([f"target.{key} = source.{key}" for key in merge_key_list])
        update_assignments = ", ".join(
        [
            f"target.{column} = source.{column}" for column in updated_df.columns if column != "insert_timestamp"

        ])
        insert_columns = ", ".join([f"{column}" for column in updated_df.columns])
        insert_values = ", ".join([f"source.{column}" for column in updated_df.columns])

        print(f"  Merge condition    : {merge_condition}")
        print(f"  Update assignments : {update_assignments}")
        print(f"  Insert columns     : {insert_columns}")
        print(f"  Insert values      : {insert_values}")

        merge_sql = f"""
                  merge into {target_table} as target
                  using {source_view} as source
                  on {merge_condition}
                  when matched then update set {update_assignments}
                  when not matched then insert ({insert_columns}) 
                  values ({insert_values})
                  """
        print("  MERGE query to be executed:")
        print(merge_sql)

        merge_result = spark.sql(merge_sql)
        print("  MERGE completed. Metrics:")
        merge_result.show(truncate=False)
else:
    print(f"  ERROR: Unsupported load type '{normalized_load_type}'.")
    raise ValueError("load_type must be one of: FULL, APPEND, MERGE")
        
 

In [0]:
record_count = updated_df.count()
print("=" * 80)
print("[STEP 5] Finished")
print("=" * 80)
print(f"  Records processed for {target_table}: {record_count}")
print(f"  Exiting notebook with value: {record_count}")
dbutils.notebook.exit(str(record_count))